In [15]:
import pandas as pd

df = pd.read_csv("../data/milestone1_output_MisaKanaujiya.csv")
df.head()


,body,predicted_triage
0,Reminder: The client meeting is scheduled at 1...,respond_or_act
1,Your invoice of INR 25515.09 is due on 2025-12...,respond_or_act
2,Reminder: The client meeting is scheduled at 1...,respond_or_act
3,"Hello team, please find the attached weekly re...",respond_or_act
4,"Hello team, please find the attached weekly re...",respond_or_act


In [16]:
source_col = 'body'   
df['clean_text'] = (
    df[source_col]
      .astype(str)
      .str.lower()
      .str.replace('[^a-zA-Z ]', '', regex=True)
)
df[[source_col, 'clean_text']].head()


,body,clean_text
0,Reminder: The client meeting is scheduled at 1...,reminder the client meeting is scheduled at t...
1,Your invoice of INR 25515.09 is due on 2025-12...,your invoice of inr is due on please pay to ...
2,Reminder: The client meeting is scheduled at 1...,reminder the client meeting is scheduled at t...
3,"Hello team, please find the attached weekly re...",hello team please find the attached weekly rep...
4,"Hello team, please find the attached weekly re...",hello team please find the attached weekly rep...


In [17]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words("english"))

df['keywords'] = df['clean_text'].apply(
    lambda x: [w for w in x.split() if w not in stop_words]
)

df[['clean_text', 'keywords']].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\barat\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,clean_text,keywords
0,reminder the client meeting is scheduled at t...,"[reminder, client, meeting, scheduled, tomorro..."
1,your invoice of inr is due on please pay to ...,"[invoice, inr, due, please, pay, avoid, late, ..."
2,reminder the client meeting is scheduled at t...,"[reminder, client, meeting, scheduled, tomorro..."
3,hello team please find the attached weekly rep...,"[hello, team, please, find, attached, weekly, ..."
4,hello team please find the attached weekly rep...,"[hello, team, please, find, attached, weekly, ..."


In [18]:
def triage_rule(text):
    text = text.lower()

    if "refund" in text or "urgent" in text or "password" in text:
        return "notify_human"

    if "newsletter" in text or "promotion" in text:
        return "ignore"

    return "respond_or_act"

df['triage'] = df['clean_text'].apply(triage_rule)
df[['body', 'triage']].head()

,body,triage
0,Reminder: The client meeting is scheduled at 1...,respond_or_act
1,Your invoice of INR 25515.09 is due on 2025-12...,respond_or_act
2,Reminder: The client meeting is scheduled at 1...,respond_or_act
3,"Hello team, please find the attached weekly re...",respond_or_act
4,"Hello team, please find the attached weekly re...",respond_or_act


In [19]:
df.to_csv("../data/milestone1_output_MisaKanaujiya.csv", index=False)

In [20]:
# Check columns
print(df.columns)

# Apply triage rule
df['predicted_triage'] = df['clean_text'].apply(triage_rule)

# View result
df[['body', 'predicted_triage']].head()


Index(['body', 'predicted_triage', 'clean_text', 'keywords', 'triage'], dtype='object')


,body,predicted_triage
0,Reminder: The client meeting is scheduled at 1...,respond_or_act
1,Your invoice of INR 25515.09 is due on 2025-12...,respond_or_act
2,Reminder: The client meeting is scheduled at 1...,respond_or_act
3,"Hello team, please find the attached weekly re...",respond_or_act
4,"Hello team, please find the attached weekly re...",respond_or_act


In [21]:
df[['body', 'predicted_triage']].to_csv("../data/milestone1_output_MisaKanaujiya.csv", index=False)


In [22]:
import pandas as pd
df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()

,id,sender,subject,body,priority,triage_label
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond


In [23]:
#Select 100 test emails

eval_df = df.sample(n=100, random_state=42)
eval_df = eval_df.reset_index(drop=True)
eval_df.head()

,id,sender,subject,body,priority,triage_label
0,96,no-reply@service.com,Weekly Newsletter,Your order #3634 has been shipped and is expec...,low,ignore
1,16,news@techblog.com,Payment Overdue,"Hi, don't miss our sale with discounts up to 7...",low,notify_human
2,31,news@techblog.com,Weekly Newsletter,Notice: Your account will be locked unless ver...,high,notify_human
3,159,no-reply@service.com,Welcome to Service,Reminder: The client meeting is scheduled at 9...,low,respond
4,129,sales@shop.com,Invoice Due,Your order #6464 has been shipped and is expec...,low,respond


In [24]:
#create a column of ideal_response(if elif logic)

def ideal_response(text):
    """
    Returns the ideal response based on email content.
    """
    t = str(text).lower()  # ensure lowercase for matching

    # Security-related emails
    if any(k in t for k in ['password', 'reset password', 'security issue']):
        return "Please escalate this security issue to the IT team immediately."

    # Marketing / promotion emails
    elif any(k in t for k in ['promotion', 'sale', 'offer', 'unsubscribe', 'newsletter']):
        return "No action needed. This is a promotional email."

    # Payment / invoice / meeting emails
    elif any(k in t for k in ['invoice', 'payment', 'overdue', 'due on', 'meeting']):
        return "Please respond appropriately to the client regarding payment or schedule."

    # Default response
    else:
        return "Please read the email and respond as necessary."

In [29]:
# ensure clean_text exists before applying ideal_response
if 'clean_text' not in df.columns:
    df['clean_text'] = (
        df['body'].astype(str)
             .str.lower()
             .str.replace('[^a-zA-Z ]', '', regex=True)
    )

df['ideal_response'] = df['clean_text'].apply(ideal_response)
# use the existing triage_label column (not 'triage')
df[['clean_text', 'triage_label', 'ideal_response']].head()

,clean_text,triage_label,ideal_response
0,reminder the client meeting is scheduled at t...,notify_human,Please respond appropriately to the client reg...
1,your invoice of inr is due on please pay to ...,respond,Please respond appropriately to the client reg...
2,reminder the client meeting is scheduled at t...,ignore,Please respond appropriately to the client reg...
3,hello team please find the attached weekly rep...,respond,Please read the email and respond as necessary.
4,hello team please find the attached weekly rep...,respond,Please read the email and respond as necessary.
